In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import get_isc_catalog, catalog_to_dataframe
from tqdm.auto import tqdm
from pathlib import Path
from torch.utils.data import Dataset
from typing import Optional, Union
import re

class MultiCatalogDataset(Dataset):
    def __init__(self, root_dir : Union[str, Path], in_memory : bool = True, filename_pattern : str = '*.csv', event_id_name : str = 'event_id', time_name : str = 'time', latitude_name : str = 'latitude', longitude_name : str = 'longitude', depth_name : str = 'depth', magnitude_name : str = 'magnitude', mag_root_dir : Optional[Union[str, Path]] = None, magnitude_type : str = 'magnitude_type', magnitude_family : str = "magnitude_family"):
        self.in_memory = in_memory
        self.event_id_name = event_id_name
        self.time_name = time_name
        self.latitude_name = latitude_name
        self.longitude_name = longitude_name
        self.depth_name = depth_name
        self.magnitude_name = magnitude_name
        self.magnitude_type = magnitude_type
        self.magnitude_family = magnitude_family
        self.root_dir = Path(root_dir) if isinstance(root_dir, str) else root_dir
        if mag_root_dir is not None:
            self.mag_root_dir = Path(mag_root_dir) if isinstance(mag_root_dir, str) else mag_root_dir
        else:
            self.mag_root_dir = None
        self.catalog_files = list(self.root_dir.glob(filename_pattern))
        if self.mag_root_dir is not None:
            self.magnitude_files = list(Path(mag_root_dir).glob(filename_pattern)) if mag_root_dir is not None else []
        else:
            self.magnitude_files = []


        if self.in_memory:
            self.loaded_catalogs = {}
        else:
            self.loaded_catalogs = None
        self.cat_metadatas = []
        for file in self.catalog_files:
            if not file.is_file():
                continue
            file_key = file.stem
            df = pd.read_csv(file, parse_dates=[self.time_name])
            #df.sort_values(by=self.time_name, inplace=True)
            min_time = df[self.time_name].min().to_numpy()
            max_time = df[self.time_name].max().to_numpy()
            min_lat = df[self.latitude_name].min()
            max_lat = df[self.latitude_name].max()
            min_lon = df[self.longitude_name].min()
            max_lon = df[self.longitude_name].max()
            mean_mag = df[self.magnitude_name].mean()
            max_mag = df[self.magnitude_name].max()
            num_events = len(df)

            if self.in_memory:
                self.loaded_catalogs[file_key] = df
            
            self.cat_metadatas.append({"name" : file_key,
                "min_time": min_time,
                "max_time": max_time,
                "min_lat": min_lat,
                "max_lat": max_lat,
                "min_lon": min_lon,
                "max_lon": max_lon,
                "mean_mag" : mean_mag,
                "max_mag" : max_mag,
                "num_events" : num_events
            })
        self.cat_metadatas = pd.DataFrame(self.cat_metadatas)
        
    def _get_relevant_keys(self, start_time : Union[str, np.datetime64], end_time : Union[str, np.datetime64],
                           min_lat : float = -np.inf, max_lat : float = np.inf,
                           min_lon : float = -np.inf, max_lon : float = np.inf):
        if isinstance(start_time, str):
            start_time = np.datetime64(start_time)
        if isinstance(end_time, str):
            end_time = np.datetime64(end_time)
        relevant_df = self.cat_metadatas[(self.cat_metadatas['min_time'] >= start_time) & (self.cat_metadatas['max_time'] < end_time)]
        relevant_df = relevant_df[(relevant_df['min_lat'] >= min_lat) & (relevant_df['max_lat'] <= max_lat)]
        relevant_df = relevant_df[(relevant_df['min_lon'] >= min_lon) & (relevant_df['max_lon'] < max_lon)]
        return relevant_df['name'].tolist()

    def _load_data(self, start_time : Union[str, np.datetime64], end_time : Union[str, np.datetime64],
                           min_lat : float = -np.inf, max_lat : float = np.inf,
                           min_lon : float = -np.inf, max_lon : float = np.inf):
        relevant_keys = self._get_relevant_keys(start_time, end_time, min_lat, max_lat, min_lon, max_lon)
        sub_dfs = []
        if len(relevant_keys) > 0:
            for rkey in relevant_keys:
                if self.in_memory:
                    df = self.loaded_catalogs[rkey]
                else:
                    df = pd.read_csv(self.catalog_files[rkey], parse_dates=[self.time_name])

                sub_df = df[(df[self.time_name] >= start_time) & (df[self.time_name] < end_time)]
                sub_df = sub_df[(sub_df[self.latitude_name] >= min_lat) & (sub_df[self.latitude_name] <= max_lat)]
                sub_df = sub_df[(sub_df[self.longitude_name] >= min_lon) & (sub_df[self.longitude_name] < max_lon)]
                if len(sub_df) > 0:
                    sub_dfs.append(sub_df)
            sub_dfs = pd.concat(sub_dfs)
            return sub_dfs
        else:
            return None

    def __len__(self):
        return self.cat_metadatas['num_events'].sum()

In [9]:
mc_dataset = MultiCatalogDataset(root_dir="catalogs/isc/", filename_pattern="japan*.csv", in_memory=True)

In [20]:
mc_dataset._load_data("2012-01-01", "2020-01-01").groupby('magnitude_type').count()

,event_id,time,latitude,longitude,depth,magnitude,magnitude_family
magnitude_type,,,,,,,
M,15594,15594,15594,15594,15594,15594,15594
MD,39,39,39,39,39,39,39
ML,18,18,18,18,18,18,18
MS,2761,2761,2761,2761,2761,2761,2761
MV,8320,8320,8320,8320,8320,8320,8320
MW,981,981,981,981,981,981,981
mb,16977,16977,16977,16977,16977,16977,16977
mb1,1,1,1,1,1,1,1
mbtmp,2,2,2,2,2,2,2


In [18]:
def datetime64_to_times(datetimes):
    """
    Convert np.datetime64 scalar/array to:
        [..., 6] = [second, minute, hour, day_of_year, month, year]

    Conventions:
        second      : [0, 60), fractional seconds preserved
        minute      : 0..59
        hour        : 0..23
        day_of_year : 0..364/365
        month       : 0..11
        year        : e.g. 2026
    """
    dt = np.asarray(datetimes).astype("datetime64[ns]")

    year = dt.astype("datetime64[Y]")
    month = dt.astype("datetime64[M]")
    day = dt.astype("datetime64[D]")
    hour = dt.astype("datetime64[h]")
    minute = dt.astype("datetime64[m]")

    years = year.astype(np.int64) + 1970

    months = (
        month.astype(np.int64)
        - year.astype("datetime64[M]").astype(np.int64)
    )

    days_of_year = (
        day - year.astype("datetime64[D]")
    ).astype("timedelta64[D]").astype(np.int64)

    hours = (
        hour - day.astype("datetime64[h]")
    ).astype("timedelta64[h]").astype(np.int64)

    minutes = (
        minute - hour.astype("datetime64[m]")
    ).astype("timedelta64[m]").astype(np.int64)

    # Fractional seconds preserved
    seconds = (
        (dt - minute.astype("datetime64[ns]"))
        / np.timedelta64(1, "s")
    )

    return np.stack(
        [
            seconds,
            minutes,
            hours,
            days_of_year,
            months,
            years,
        ],
        axis=-1,
    ).astype(np.float32)